# Initial user profiles (1-second chunks)

Visualizes the output of `experiments/scripts/build_initial_profiles.py`, which ports the idea from `experiments/initial_program/`:

1. Split each session into 1-second chunks. The position at second *t* is the first event with `record timestamp` >= *t*.
2. Shift each chunk so point *t* sits at (0, 0); point *t*+1 becomes the displacement.
3. Mark **burn-in** chunks: the first movement of a session, or the first movement after a long idle.
4. Build each user's profile from 3 random training sessions: chunk endpoints → convex hull → concave hull.

Run the script first:

```bash
python experiments/scripts/build_initial_profiles.py
```


In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = next(parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "outputs").is_dir())
PROFILE_DIR = PROJECT_ROOT / "outputs" / "balabit_initial_profiles"

profiles, chunks = {}, {}
for user_dir in sorted(PROFILE_DIR.iterdir(), key=lambda path: int(path.name[4:]) if path.name.startswith("user") else 0):
    if not (user_dir / "profile.json").exists():
        continue
    profiles[user_dir.name] = json.loads((user_dir / "profile.json").read_text())
    chunks[user_dir.name] = pd.read_csv(user_dir / "chunks.csv")

pd.DataFrame(
    {
        "sessions": [", ".join(p["sessions"]) for p in profiles.values()],
        "chunks": [p["chunk_count"] for p in profiles.values()],
        "moving": [p["moving_count"] for p in profiles.values()],
        "burn_in": [p["burn_in_count"] for p in profiles.values()],
        "concave_area_px2": [round(p["concave_hull"]["area"]) for p in profiles.values()],
    },
    index=list(profiles),
)


## All users: chunk endpoints and hulls

Blue = moving chunks, orange = burn-in chunks, gray = convex hull, red = concave hull. The y-axis is inverted to match screen coordinates.


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(22, 9.5), sharex=True, sharey=True)
for ax, (user_id, profile) in zip(axes.flat, profiles.items()):
    frame = chunks[user_id]
    moving, burn_in = frame[frame.is_moving], frame[frame.is_burn_in]
    ax.scatter(moving.t_plus_1_x_rel, moving.t_plus_1_y_rel, s=2, alpha=0.15, color="tab:blue", label="moving chunk")
    ax.scatter(burn_in.t_plus_1_x_rel, burn_in.t_plus_1_y_rel, s=5, alpha=0.6, color="tab:orange", label="burn-in chunk")
    for hull, color, label in [("convex_hull", "gray", "convex hull"), ("concave_hull", "tab:red", "concave hull")]:
        coordinates = profile[hull]["coordinates"]
        if coordinates:
            ax.plot(*zip(*coordinates), color=color, lw=1.2, label=label)
    ax.set_title(f"{user_id}: {profile['moving_count']:,} moving, {profile['burn_in_count']} burn-in", fontsize=10)
    ax.set_aspect("equal")
    ax.axhline(0, color="k", lw=0.3)
    ax.axvline(0, color="k", lw=0.3)
axes.flat[0].invert_yaxis()
axes.flat[0].legend(loc="lower left", fontsize=8, markerscale=3)
fig.tight_layout()
plt.show()


## One user: starburst of chunk vectors

Same view as the single-graph cell in `initial_program`: every chunk drawn as a line from (0, 0), colored by time within its session.


In [ ]:
from matplotlib.collections import LineCollection

USER_ID = "user15"

frame = chunks[USER_ID][chunks[USER_ID].is_moving]
segments = [[(0, 0), (x, y)] for x, y in zip(frame.t_plus_1_x_rel, frame.t_plus_1_y_rel)]

fig, ax = plt.subplots(figsize=(9, 9))
lines = LineCollection(segments, cmap="viridis", linewidths=0.6, alpha=0.5)
lines.set_array(frame.t.to_numpy())
ax.add_collection(lines)
ax.plot(*zip(*profiles[USER_ID]["concave_hull"]["coordinates"]), color="tab:red", lw=1.5, label="concave hull")
ax.autoscale()
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_title(f"{USER_ID}: {len(frame):,} moving chunks from {len(profiles[USER_ID]['sessions'])} sessions")
ax.set_xlabel("x relative to t")
ax.set_ylabel("y relative to t")
ax.legend()
fig.colorbar(lines, ax=ax, label="t (seconds into session)")
plt.show()
